1. Ingestão e Organização dos Dados
Inclui:
estrutura de pastas;
SparkSession;
leitura dos CSVs;
schema;
validação;
amostragem;
criação dos DataFrames.
2. Entendimento Inicial dos Dados
SOMENTE a parte estrutural:

Inclua:

número de linhas/colunas;
tipos de dados;
classificação de variáveis;
análise de nulos;
baixa variabilidade;
inconsistências estruturais.

NÃO faça EDA visual aqui.

4. Limpeza dos Dados
Tudo aqui.

Inclua:

tratamento de nulos;
remoção de colunas;
correção de inconsistências;
tratamento de outliers;
padronização de tipos.
5. Engenharia de Atributos
Tudo aqui.

Inclua:

criação de ratios;
idade;
tempo de emprego;
flags;
agregações bureau;
joins;
feature engineering.
Final do notebook

Salvar dataset tratado:

df_featured.write.mode("overwrite").parquet(
    "data/processed/home_credit_featured.parquet"
)

# Ingestão de Dados

In [1]:
# Importando Bibliotecas
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, when, count, avg, abs as spark_abs, column

In [2]:
# Iniciando sessão Spark
spark = SparkSession.builder \
    .appName("Studys") \
    .master("local[*]") \
    .getOrCreate()

print("SparkSession iniciada com sucesso")

SparkSession iniciada com sucesso


In [3]:
# Lendo as tabela e salvando em dfs

# Base principal (contém dados cadastrais, financeiros e TARGET)
application_train = r'DataSets/application_train.csv'
df_application_train = spark.read.csv(
    application_train,
    header=True,
    inferSchema=False,
    sep=","
)

# Histórico de crédito externo (Contém histórico em outros bancos, dividas passadas, inadimplência anterior)
bureau_csv = r"DataSets/bureau.csv"
df_bureau = spark.read.csv(
    bureau_csv,
    header=True,
    inferSchema=False,
    sep=","
)

# Histórico interno (contém pedidos anteriores, aprovações e recusas, compartamento histórico)
previous_application = r'DataSets/previous_application.csv'
df_previous_application = spark.read.csv(
    previous_application,
    header=True,
    inferSchema=False,
    sep=","
)

In [4]:
print("Application shape:", "Quantidade de dados ",df_application_train.count(), "Qtdd colunas ",len(df_application_train.columns))
print("Bureau shape:", "Quantidade de dados ", df_bureau.count(), "Qtdd colunas ", len(df_bureau.columns))
print("Previous shape:", "Quantidade de dados ", df_previous_application.count(), "Qtdd colunas ", len(df_previous_application.columns))

Application shape: Quantidade de dados  307511 Qtdd colunas  122
Bureau shape: Quantidade de dados  1716428 Qtdd colunas  17
Previous shape: Quantidade de dados  1670214 Qtdd colunas  37


# Data Understanding

In [6]:
# Selecionando colunas Relevantes
col_application_train = [
    "SK_ID_CURR"            # ID
    , "TARGET"              # variável alvo → 1 = inadimplente (default), 0 = adimplente
    , "CODE_GENDER"         # categórica nominal, gênero do cliente (M/F)
    , "FLAG_OWN_CAR"        # binária categórica, possui carro (Y/N)
    , "FLAG_OWN_REALTY"     # binária categórica, possui imóvel (Y/N)
    , "CNT_CHILDREN"        # numérica discreta (contagem), número de filhos
    , "AMT_INCOME_TOTAL"    # numérica contínua, renda total do cliente
    , "AMT_CREDIT"          # numérica contínua, valor do crédito solicitado
    , "AMT_ANNUITY"         # numérica contínua, valor da anuidade/prestação do crédito
    , "AMT_GOODS_PRICE"     # numérica contínua, preço dos bens financiados
    ,"NAME_INCOME_TYPE"     # categórica nominal, tipo de renda (trabalho, aposentado, etc.)
    , "NAME_EDUCATION_TYPE" # categórica ordinal, nível educacional
    , "NAME_FAMILY_STATUS"  # categórica nominal, estado civil
    , "NAME_HOUSING_TYPE"   # categórica nominal, tipo de moradia
    , "DAYS_BIRTH"          # numérica contínua, idade em dias (valor negativo → dias antes da aplicação)
    # Observação: geralmente convertido para anos
    , "DAYS_EMPLOYED"       # numérica contínua, dias empregado
    , "OCCUPATION_TYPE"     # categórica nominal, ocupação profissional
]

col_bureau = [
    "SK_ID_CURR"            # ID, FK
    , "SK_ID_BUREAU"        # ID, identificador único de cada registro de crédito reportado por bureaus externos (um cliente pode ter vários).
    , "CREDIT_ACTIVE"       # categórica nominal, status atual do crédito no bureau (Active, Closed, Sold, Bad debt)
    , "AMT_CREDIT_SUM"      # numérica contínua, valor total do crédito reportado pelo bureau para aquele contrato
    , "AMT_CREDIT_SUM_OVERDUE" # numérica contínua, valor em atraso (overdue). Indica inadimplência naquele crédito específico.
    , "DAYS_CREDIT"         # numérica discreta, número de dias desde a concessão do crédito até a data atual (quando negativo indicando passado).
]

col_previous_application = [
    "SK_ID_CURR"            # ID, FK
    , "SK_ID_PREV"          # ID
    , "NAME_CONTRACT_STATUS" # categórica nominal, status da aplicação anterior (Approved, Refused, Canceled, Unused offer)
    , "AMT_APPLICATION"     # numérica contínua, valor solicitado pelo cliente na aplicação anterior.
    , "AMT_CREDIT"          # numérica contínua, valor efetivamente concedido
    , "AMT_ANNUITY"         # numérica contínua, valor da parcela periódica do crédito.
    , "DAYS_DECISION"       # numérica discreta, dias desde a decisão da aplicação anterior até o momento atual (tipicamente negativo).
]

df_application_train = df_application_train.select(col_application_train)
df_bureau = df_bureau.select(col_bureau)
df_previous_application = df_previous_application.select(col_previous_application)

In [6]:
print("Application shape:", "Quantidade de dados ",df_application_train.count(), "Qtdd colunas ",len(df_application_train.columns))
print("Bureau shape:", "Quantidade de dados ", df_bureau.count(), "Qtdd colunas ", len(df_bureau.columns))
print("Previous shape:", "Quantidade de dados ", df_previous_application.count(), "Qtdd colunas ", len(df_previous_application.columns))

Application shape: Quantidade de dados  307511 Qtdd colunas  17
Bureau shape: Quantidade de dados  1716428 Qtdd colunas  6
Previous shape: Quantidade de dados  1670214 Qtdd colunas  7


In [7]:
# Identificação de linhas Nulas por Coluna - df_application_train

total_rows = df_application_train.count()

null_counts = (
    df_application_train
    .select([
        count(when(col(column).isNull(), column)).alias(column)
        for column in df_application_train.columns
    ]).toPandas()
    .T # Transpose, transforma linhas em colunas
    .reset_index() # transforma o índice em uma coluna de cabeçalho da tb
)

null_counts.columns = ["coluna", "qtdd_nulos"] # Renomeia as colunas

null_counts["percentual (%)"] = (
    null_counts["qtdd_nulos"] / total_rows * 100
).round(2)

null_counts = (
    null_counts[null_counts["qtdd_nulos"] > 0]
    .sort_values(by="qtdd_nulos", ascending=False)
)

print("======== Colunas com valores nulos  ========")
print("========== df_application_train ===========")
print(null_counts.to_string(index=False))


======== Colunas com valores nulos  ========
========== df_application_train ===========
         coluna  qtdd_nulos  percentual (%)
OCCUPATION_TYPE       96391           31.35
AMT_GOODS_PRICE         278            0.09
    AMT_ANNUITY          12            0.00


In [8]:
# Identificação de linhas Nulas por Coluna - df_bureau

total_rows = df_bureau.count()

null_counts = (
    df_bureau
    .select([
        count(when(col(column).isNull(), column)).alias(column)
        for column in df_bureau.columns
    ]).toPandas()
    .T # Transpose, transforma linhas em colunas
    .reset_index() # transforma o índice em uma coluna de cabeçalho da tb
)

null_counts.columns = ["coluna", "qtdd_nulos"] # Renomeia as colunas

null_counts["percentual (%)"] = (
    null_counts["qtdd_nulos"] / total_rows * 100
).round(2)

null_counts = (
    null_counts[null_counts["qtdd_nulos"] > 0]
    .sort_values(by="qtdd_nulos", ascending=False)
)

print("======== Colunas com valores nulos  ========")
print("================ df_bureau =================")
print(null_counts.to_string(index=False))


======== Colunas com valores nulos  ========
================ df_bureau =================
        coluna  qtdd_nulos  percentual (%)
AMT_CREDIT_SUM          13             0.0


In [9]:
# Identificação de linhas Nulas por Coluna - df_previous_application

total_rows = df_previous_application.count()

null_counts = (
    df_previous_application
    .select([
        count(when(col(column).isNull(), column)).alias(column)
        for column in df_previous_application.columns
    ]).toPandas()
    .T # Transpose, transforma linhas em colunas
    .reset_index() # transforma o índice em uma coluna de cabeçalho da tb
)

null_counts.columns = ["coluna", "qtdd_nulos"] # Renomeia as colunas

null_counts["percentual (%)"] = (
    null_counts["qtdd_nulos"] / total_rows * 100
).round(2)

null_counts = (
    null_counts[null_counts["qtdd_nulos"] > 0]
    .sort_values(by="qtdd_nulos", ascending=False)
)

print("======== Colunas com valores nulos  ========")
print("========= df_previous_application ==========")
print(null_counts.to_string(index=False))


======== Colunas com valores nulos  ========
========= df_previous_application ==========
     coluna  qtdd_nulos  percentual (%)
AMT_ANNUITY      372235           22.29
 AMT_CREDIT           1            0.00


In [10]:
# Dados com incosistência (Essa análise foi realizada com base na documentaçõo do repositório com Datasets do Kaggle)
# Foi identificado uma incosistência na coluna DAYS_EMPLOYED do df  df_application_train.
# O valor 365243 aparece quando o cliente está desempregado, ele representa um valor aplicado errado na origem

total_rows = df_application_train.count()

qtd_inc = df_application_train.filter(col("DAYS_EMPLOYED") == "365243").count()
percentual = round(qtd_inc / total_rows * 100, 2)

print("=== Inconsistências em DAYS_EMPLOYED ===")
print(f"Registros com valor 365243 (representa 'desempregado' codificado incorretamente): {qtd_inc}")
print(f"Representa {percentual}% dos registros")

=== Inconsistências em DAYS_EMPLOYED ===
Registros com valor 365243 (representa 'desempregado' codificado incorretamente): 55374
Representa 18.01% dos registros


# Data Cleaning

In [21]:
# Convertendo colunas numéricas de string para double para serem analisadas
# Corrigindo a coluna DAYS_EMPLOYED == 365243, porque ela representa um código para "desempregado" → substituir por null

df_eda = df_application_train\
    .withColumn("AMT_INCOME_TOTAL", col("AMT_INCOME_TOTAL").cast("double"))\
    .withColumn("AMT_CREDIT", col("AMT_CREDIT").cast("double"))\
    .withColumn("AMT_ANNUITY", col("AMT_ANNUITY").cast("double"))\
    .withColumn("AMT_GOODS_PRICE", col("AMT_GOODS_PRICE").cast("double"))\
    .withColumn("CNT_CHILDREN", col("CNT_CHILDREN").cast("double"))\
    .withColumn("DAYS_BIRTH", col("DAYS_BIRTH").cast("int"))\
    .withColumn(
        "DAYS_EMPLOYED",
        when(col("DAYS_EMPLOYED") == 365243, None)
        .otherwise(col("DAYS_EMPLOYED"))
        .cast("double")
    )\
    .withColumn("TARGET", col("TARGET").cast("int"))

# Tratamento de valores nulos
# Colunas numéricas
colunas_numericas = [
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "DAYS_EMPLOYED"
]

# Preenchendo nulos com a média
for c in colunas_numericas:
    media = df_eda.agg(avg(c)).collect()[0][0]
    df_eda = df_eda.fillna({c: round(media, 2)})

# Colunas categóricas
df_eda = df_eda.fillna({
    "OCCUPATION_TYPE": "Unknown"
})

print("=== Limpeza concluída ===")
print(f"Total de linhas: {df_eda.count()}")

# Verificar nulos restantes
null_after = df_eda.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_eda.columns
]).toPandas().T.reset_index()

null_after.columns = ["coluna", "nulos"]
null_after = null_after[null_after["nulos"] > 0]

if len(null_after) == 0:
    print("Nenhum valor nulo restante!")
else:
    print("\nNulos restantes:")
    print(null_after.to_string(index=False))

=== Limpeza concluída ===
Total de linhas: 307511
Nenhum valor nulo restante!


# Exploratory Data Analysis

In [22]:
print("====================== Análise Univariada ======================")
df_eda.select(
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY"
).describe().show()

====================== Análise Univariada ======================
+-------+-----------------+-----------------+------------------+
|summary| AMT_INCOME_TOTAL|       AMT_CREDIT|       AMT_ANNUITY|
+-------+-----------------+-----------------+------------------+
|  count|           307511|           307511|            307511|
|   mean|168797.9192969845|599025.9997057016|27108.573909030893|
| stddev| 237123.146278856|402490.7769958547|14493.454516925443|
|    min|          25650.0|          45000.0|            1615.5|
|    max|           1.17E8|        4050000.0|          258025.5|
+-------+-----------------+-----------------+------------------+



In [13]:
# Análise da variável TARGET

target_pd = df_application_train.groupBy("TARGET").count().orderBy("TARGET").toPandas()
total = target_pd["count"].sum()
target_pd["percentual (%)"] = (target_pd["count"] / total * 100).round(2)
target_pd["label"] = target_pd["TARGET"].map({"0": "Adimplente (pagou)", "1": "Inadimplente (ñ pagou)"})

print("============= Análise do TARGET =============")
print(target_pd[["label", "count", "percentual (%)"]].to_string(index=False))
print(f"\nTotal: {total} registros")

============= Análise do TARGET =============
                 label  count  percentual (%)
    Adimplente (pagou) 282686           91.93
Inadimplente (ñ pagou)  24825            8.07

Total: 307511 registros


In [14]:
# Início da análise bivariada das variáveis
# Fazendo count de Renda média, crédito médio e parcela média por caso de adimplência e inadimplência.

from pyspark.sql.functions import avg, round as spark_round

renda_credito = df_eda.groupBy("TARGET").agg(
    spark_round(avg("AMT_INCOME_TOTAL"), 2).alias("renda_media"),
    spark_round(avg("AMT_CREDIT"), 2).alias("credito_medio"),
    spark_round(avg("AMT_ANNUITY"), 2).alias("parcela_media")
).withColumn(
    "TARGET",
    when(col("TARGET") == 0, "Adimplente")
    .otherwise("Inadimplente")
).orderBy("TARGET").toPandas()


print("====== Renda, Crédito e Parcela média por TARGET ======")
print(renda_credito.to_string(index=False))


====== Renda, Crédito e Parcela média por TARGET ======
      TARGET  renda_media  credito_medio  parcela_media
  Adimplente    169077.72      602648.28       27163.62
Inadimplente    165611.76      557778.53       26481.74


In [15]:
# Análise bivariada com variaveis categóricas
# Idade média por TARGET
# DAYS_BIRTH é negativo: -9461 significa que o cliente nasceu 9461 dias antes da solicitação
idade = df_eda.withColumn("idade_anos", (spark_abs(col("DAYS_BIRTH")) / 365).cast("int")) \
    .groupBy("TARGET").agg(
        spark_round(avg("idade_anos"), 1).alias("idade_media")
    ).orderBy("TARGET").toPandas()

idade["TARGET"] = idade["TARGET"].map({0: "Adimplente", 1: "Inadimplente"})
print("=== Idade média por TARGET ===")
print(idade.to_string(index=False))

=== Idade média por TARGET ===
      TARGET  idade_media
  Adimplente         43.7
Inadimplente         40.3


In [16]:
# Taxa de inadimplência por gênero
print("=== Taxa de inadimplência por Gênero ===")
df_eda.groupBy("CODE_GENDER").agg(
    count("*").alias("total"),
    spark_round(avg(col("TARGET").cast("double")) * 100, 2).alias("taxa_inadimplencia (%)")
).orderBy("taxa_inadimplencia (%)").show()

=== Taxa de inadimplência por Gênero ===
+-----------+------+----------------------+
|CODE_GENDER| total|taxa_inadimplencia (%)|
+-----------+------+----------------------+
|        XNA|     4|                   0.0|
|          F|202448|                   7.0|
|          M|105059|                 10.14|
+-----------+------+----------------------+



In [17]:
# Inadimplência por tipo de renda
print("=== Taxa de inadimplência por Tipo de Renda ===")
df_eda.groupBy("NAME_INCOME_TYPE").agg(
    count("*").alias("total"),
    spark_round(avg(col("TARGET").cast("double")) * 100, 2).alias("taxa_inadimplencia (%)")
).orderBy("taxa_inadimplencia (%)", ascending=False).show()


=== Taxa de inadimplência por Tipo de Renda ===
+--------------------+------+----------------------+
|    NAME_INCOME_TYPE| total|taxa_inadimplencia (%)|
+--------------------+------+----------------------+
|     Maternity leave|     5|                  40.0|
|          Unemployed|    22|                 36.36|
|             Working|158774|                  9.59|
|Commercial associate| 71617|                  7.48|
|       State servant| 21703|                  5.75|
|           Pensioner| 55362|                  5.39|
|             Student|    18|                   0.0|
|         Businessman|    10|                   0.0|
+--------------------+------+----------------------+



In [18]:
# Inadimplência por escolaridade
print("=== Taxa de inadimplência por Escolaridade ===")
df_eda.groupBy("NAME_EDUCATION_TYPE").agg(
    count("*").alias("total"),
    spark_round(avg(col("TARGET").cast("double")) * 100, 2).alias("taxa_inadimplencia (%)")
).orderBy("taxa_inadimplencia (%)", ascending=False).show()

=== Taxa de inadimplência por Escolaridade ===
+--------------------+------+----------------------+
| NAME_EDUCATION_TYPE| total|taxa_inadimplencia (%)|
+--------------------+------+----------------------+
|     Lower secondary|  3816|                 10.93|
|Secondary / secon...|218391|                  8.94|
|   Incomplete higher| 10277|                  8.48|
|    Higher education| 74863|                  5.36|
|     Academic degree|   164|                  1.83|
+--------------------+------+----------------------+



# Feature Engineering


In [23]:
# credit_income_ratio: quanto do crédito representa em relação à renda (risco de endividamento)
# annuity_income_ratio: quanto da parcela compromete a renda mensal
# idade_anos: converter DAYS_BIRTH (negativo, em dias) para anos positivos
# anos_empregado: converter DAYS_EMPLOYED (negativo, em dias) para anos positivos
from pyspark.sql import functions as f

df_feat = df_eda\
    .withColumn("credit_income_ratio",  f.round((col("AMT_CREDIT")  / col("AMT_INCOME_TOTAL")), 2))\
    .withColumn("annuity_income_ratio", f.round((col("AMT_ANNUITY") / col("AMT_INCOME_TOTAL")), 2))\
    .withColumn("idade_anos", (spark_abs(col("DAYS_BIRTH")) / 365).cast("int"))\
    .withColumn("anos_empregado",(spark_abs(col("DAYS_EMPLOYED")) / 365).cast("int"))


df_feat.select("SK_ID_CURR", "credit_income_ratio", "annuity_income_ratio", "idade_anos", "anos_empregado").limit(5).toPandas()

Novas variáveis criadas com sucesso.


,SK_ID_CURR,credit_income_ratio,annuity_income_ratio,idade_anos,anos_empregado
0,100002,2.01,0.12,25,1
1,100003,4.79,0.13,45,3
2,100004,2.00,0.10,52,0
3,100006,2.32,0.22,52,8
4,100007,4.22,0.18,54,8


In [24]:
# Criação de faixas de renda e idade e grupando valores contínuos em categorias

df_feat = df_feat \
    .withColumn("faixa_renda",
        when(col("AMT_INCOME_TOTAL") <  90000, "baixa")
       .when(col("AMT_INCOME_TOTAL") < 180000, "media")
       .when(col("AMT_INCOME_TOTAL") < 360000, "alta")
       .otherwise("muito_alta")
    ) \
    .withColumn("faixa_idade",
        when(col("idade_anos") < 30, "jovem")
       .when(col("idade_anos") < 45, "adulto")
       .when(col("idade_anos") < 60, "idoso")
       .otherwise("senior")
    )

In [ ]:
# Parou na parte do join com a tb bureau_cs

In [10]:
spark.stop()
print("Sessão Spark encerrada.")

Sessão Spark encerrada.
